# Text-to-Speech (TTS) — speech generation benchmark on Apple Silicon

Turns every prompt of a prompt set into speech with a model in **float16** (see the registry for three exceptions) on MLX
([`mlx-audio`](https://github.com/Blaizzy/mlx-audio)) and records, per prompt: voice, prompt length, audio length,
sample rate, time to first audio, inference rate and inference time, plus system metrics (CPU / GPU / NPU utilisation,
power, temperature) sampled by `macmon`. Whisper then transcribes each clip, and the word error rate against the
prompt shows how intelligible the speech is.

Each model is kept as a local copy at `models/<model>-<precision>/` (made once from its MLX checkpoint on Hugging
Face) and loaded from there.

**Outputs**
- `outputs/tts/<family>/<params>/<prompt set>/`: one `.wav` per prompt and `<model>_simple_metrics.json`
- `metrics_analysis/tts/<family>/<params>/<prompt set>/`: `Model_Level_Metrics/<model>_prompt_metrics.csv`
  and `System_Level_Metrics/<prompt id>_system_metrics.csv`

## 1. Settings and model registry

In [ ]:
import gc
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import mlx.core as mx
import numpy as np
import pandas as pd
import soundfile as sf

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'prompts').is_dir())   # project root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault('HF_HUB_DISABLE_IMPLICIT_TOKEN', '1')   # every model here is public
import espeakng_loader   # Kitten-TTS turns text into phonemes with espeak-ng (the copy bundled in espeakng-loader)
os.environ.setdefault('PHONEMIZER_ESPEAK_LIBRARY', espeakng_loader.get_library_path())
os.environ.setdefault('ESPEAK_DATA_PATH', espeakng_loader.get_data_path())

# family        size      params  model               MLX checkpoint on Hugging Face                        params (M)  precision
REGISTRY = [
    ('Bark',       'SMALL',  '420M', 'bark-small',       'mlx-community/bark-small',                          420,  'float32'),
    ('Bark',       'LARGE',  '1.1B', 'bark',             'mlx-community/bark',                                1144, 'float32'),
    ('Kitten-TTS', 'SMALL',  '15M',  'kitten-tts-nano',  'mlx-community/kitten-tts-nano-0.8-fp16',            15,   'float16'),
    ('Kitten-TTS', 'MEDIUM', '36M',  'kitten-tts-micro', 'mlx-community/kitten-tts-micro-0.8-fp16',           36,   'float16'),
    ('Kitten-TTS', 'LARGE',  '74M',  'kitten-tts-mini',  'mlx-community/kitten-tts-mini-0.8-fp16',            74,   'float16'),
    ('Qwen3-TTS',  'SMALL',  '0.6B', 'Qwen3-TTS-0.6B',   'mlx-community/Qwen3-TTS-12Hz-0.6B-CustomVoice-bf16', 1076, 'bfloat16'),
    ('Qwen3-TTS',  'LARGE',  '1.7B', 'Qwen3-TTS-1.7B',   'mlx-community/Qwen3-TTS-12Hz-1.7B-CustomVoice-bf16', 2087, 'float16'),
    ('OuteTTS',    'MEDIUM', '1B',   'OuteTTS-1.0-1B',   'mlx-community/Llama-OuteTTS-1.0-1B-fp16',            1248, 'float16'),
]
# Bark (both sizes) and Qwen3-TTS-0.6B keep their original precision: in float16 they produce the wrong words
# (Bark) or never stop speaking (Qwen3-TTS-0.6B). Every other model runs in float16.
MODELS = pd.DataFrame(REGISTRY, columns=['family', 'size', 'params', 'model', 'source', 'params_m', 'precision']).set_index('model')
MODELS['path'] = 'models/' + MODELS.index + '-' + MODELS.precision.map({'float16': 'fp16', 'bfloat16': 'bf16', 'float32': 'fp32'})

# Each prompt's voice follows the gender in its description (male / female; 'male' if none). Qwen3-TTS also follows
# the full description as a style instruction (tone, pace, pitch); the others can't.
VOICES = {'Bark': ('en_speaker_6', 'en_speaker_9'), 'Kitten-TTS': ('expr-voice-5-m', 'expr-voice-5-f'),
          'Qwen3-TTS': ('ryan', 'vivian'), 'OuteTTS': ('default', 'default')}   # (male, female)

# Pick the model here, or from the terminal with TTS_FAMILY / TTS_SIZE / TTS_PROMPTS / TTS_COMPARE
FAMILY = os.environ.get('TTS_FAMILY', 'Qwen3-TTS')   # Bark | Kitten-TTS | Qwen3-TTS | OuteTTS
SIZE = os.environ.get('TTS_SIZE', 'LARGE')           # SMALL | MEDIUM | LARGE (only the sizes listed)
MODEL = MODELS.query('family == @FAMILY and size == @SIZE').index[0]
PROMPT_SET = os.environ.get('TTS_PROMPTS', 'tts_prompts_described')   # prompts/tti_tts_ttt_prompts/<set>.csv
COMPARE_FAMILY = os.environ.get('TTS_COMPARE', '1') == '1'           # section 5: also run the family's other models
WHISPER = ('mlx-community/whisper-large-v3-mlx', 'models/whisper-large-v3-fp16')   # checks what the speech says
SEED = 0   # Bark, Qwen3-TTS and OuteTTS sample; a fixed seed makes runs repeatable

print(f'Selected: {FAMILY} / {SIZE} -> {MODEL} -> {MODELS.path[MODEL]}')
MODELS[MODELS.family == FAMILY][['size', 'params', 'source', 'precision', 'path']]

## 2. Prepare and load a model

The first time, the model's MLX checkpoint is downloaded and its weights are saved as float16 in `models/<model>-fp16/`
(one-time step); after that it loads from there. The `mlx-community/bark` upload stores attention as separate q/k/v
weights and config fields `mlx-audio` doesn't read, so its float16 copy is rebuilt in the layout `mlx-audio` expects.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
from mlx_audio.tts.utils import load_model as load_tts

def cast(w, dtype):
    return {k: v.astype(dtype) if mx.issubdtype(v.dtype, mx.floating) else v for k, v in w.items()}

def copy_as(src, dst, dtype):
    # Copy a model folder, casting the floating-point tensors of its weight files to dtype
    for f in Path(src).rglob('*'):
        out = Path(dst) / f.relative_to(src)
        if f.is_dir():
            continue
        out.parent.mkdir(parents=True, exist_ok=True)
        if f.suffix == '.safetensors' or f.name == 'weights.npz':   # Whisper's weights.npz is saved as weights.safetensors
            w = cast(mx.load(str(f)), dtype)
            mx.save_safetensors(str(out.with_suffix('.safetensors')), w)
        else:
            shutil.copyfile(f, out)

def build_bark(dst, dtype):
    # Bark (large): merge q/k/v into one projection, rename config fields, add the causal masks mlx-audio expects
    src = Path(snapshot_download('mlx-community/bark', token=False, allow_patterns=['*.safetensors', '*.json']))
    small = Path(snapshot_download('mlx-community/bark-small', token=False,
                                   allow_patterns=['tokenizer*', 'special_tokens_map.json', 'vocab.txt']))
    w = {**mx.load(str(src / 'model.safetensors')), **mx.load(str(src / 'fine_acoustics.safetensors'))}
    out = {}
    for k, v in w.items():
        if k.endswith('attn.q_proj.weight'):
            base = k[:-len('q_proj.weight')]
            out[base + 'att_proj.weight'] = mx.concatenate([w[base + p + '_proj.weight'] for p in 'qkv'], axis=0)
        elif not k.endswith(('attn.k_proj.weight', 'attn.v_proj.weight')):
            out[k] = v
    cfg = json.loads((src / 'config.json').read_text())
    for part in ('semantic_config', 'coarse_acoustics_config', 'fine_acoustics_config'):
        cfg[part].update(n_layer=cfg[part]['num_layers'], n_head=cfg[part]['num_heads'], n_embd=cfg[part]['hidden_size'])
    cfg.update(model_size='large', n_layer=24, n_head=16, n_embd=1024)
    block = cfg['semantic_config']['block_size']
    mask = mx.tril(mx.ones((block, block))).astype(mx.bool_).reshape(1, 1, block, block)
    for part in ('semantic', 'coarse_acoustics'):
        for i in range(cfg['semantic_config']['n_layer']):
            out[f'{part}.layers.{i}.attn.bias'] = mask
    Path(dst).mkdir(parents=True)
    mx.save_safetensors(str(Path(dst) / 'model.safetensors'), cast(out, dtype))
    (Path(dst) / 'config.json').write_text(json.dumps(cfg, indent=2))
    shutil.copy(src / 'generation_config.json', Path(dst) / 'generation_config.json')
    for f in ('tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt'):
        shutil.copy(small / f, Path(dst) / f)

def prepare(source, path, precision='float16'):
    if Path(path).exists():
        return
    print(f'Preparing {path} from {source} ({precision}), one-time step...')
    dtype = getattr(mx, precision)
    try:
        build_bark(path, dtype) if source == 'mlx-community/bark' else copy_as(snapshot_download(source, token=False), path, dtype)
    except Exception:
        shutil.rmtree(path, ignore_errors=True)   # don't leave a half-prepared model behind
        raise

def load_model(name):
    m = MODELS.loc[name]
    prepare(m.source, m.path, m.precision)
    t0 = time.perf_counter()
    model = load_tts(m.path)
    mx.eval(model.parameters())
    return model, time.perf_counter() - t0

def voice_kwargs(name, description):
    # generate() arguments selecting the voice (and, for Qwen3-TTS, the style) for one prompt
    m = MODELS.loc[name]
    female = bool(re.search(r'\bfemale\b|\bwoman\b', str(description).lower()))
    voice = VOICES[m.family][female]
    if m.family == 'Bark':
        if name == 'bark-small':   # bark-small ships each preset as three .npy prompts
            return {'voice': {k: np.load(hf_hub_download('mlx-community/bark-small', f'speaker_embeddings/{voice}_{k}.npy',
                                                         token=False)) for k in ('semantic_prompt', 'coarse_prompt', 'fine_prompt')}}
        return {'voice': hf_hub_download('mlx-community/bark', f'v2/{voice}.npz', token=False)}
    if m.family == 'Kitten-TTS':
        return {'voice': voice}
    if m.family == 'Qwen3-TTS':
        return {'voice': voice, 'lang_code': 'english', **({'instruct': description} if description else {})}
    return {'max_tokens': 4096}   # OuteTTS: one built-in speaker; its default 1,200-token cap allows only ~8 s of audio

## 3. Prompts

In [ ]:
PROMPTS = pd.read_csv(Path('prompts/tti_tts_ttt_prompts') / f'{PROMPT_SET}.csv', dtype={'id': str})
PROMPTS['id'] = PROMPTS.id.str.zfill(2)
PROMPTS['description'] = PROMPTS.get('description', pd.Series('', index=PROMPTS.index)).fillna('')
print(f'{len(PROMPTS)} prompts from {PROMPT_SET}.csv')
PROMPTS

## 4. Generate speech with the selected model

Each prompt is synthesised after one untimed warm-up prompt. Bark can only produce about 14 s per call, so it gets one
sentence at a time. Each prompt is wrapped in `metrics_monitor_start()` / `metrics_monitor_stop()` (from
`macmon_metrics/`), which saves its system metrics every 100 ms. After generation (outside the timed windows) Whisper
transcribes every clip.

In [ ]:
import macmon_metrics.metrics_monitor as mm
import mlx_whisper
from macmon_metrics.metrics_monitor import metrics_monitor_start, metrics_monitor_stop, setup_output_directory
from mlx_whisper.transcribe import ModelHolder
from text_metrics import word_errors

if mm._tracker is None:   # start macmon now, so every prompt's capture has samples from its first moment
    mm._tracker = mm.MacMonTracker(binary_path=mm._resolve_macmon_binary())
    mm._tracker.start()
    time.sleep(1.5)

def synthesize(model, name, text, voice):
    # Returns (waveform, sample rate, seconds, seconds to first audio)
    mx.random.seed(SEED)
    pieces = [p for p in re.split(r'(?<=[.!?])\s*(?=[A-Z(])', text.strip()) if p.strip()] \
        if MODELS.family[name] == 'Bark' else [text]
    chunks, first, t0 = [], None, time.perf_counter()
    for piece in pieces:
        for r in model.generate(text=piece, **voice):
            first = first or time.perf_counter() - t0
            chunks.append(np.array(r.audio, dtype=np.float32).reshape(-1))
            rate = r.sample_rate
    return np.concatenate(chunks), rate, time.perf_counter() - t0, first

def transcribe_all(files):
    # Whisper (float16) transcribes each generated clip, then is unloaded
    prepare(*WHISPER)
    texts = [mlx_whisper.transcribe(f, path_or_hf_repo=WHISPER[1], language='en', verbose=None,
                                    condition_on_previous_text=False)['text'].strip() for f in files]
    ModelHolder.model, ModelHolder.model_path = None, None
    gc.collect()
    mx.clear_cache()
    return texts

def run_model(name):
    # Synthesise every prompt with one model; save audio, per-prompt metrics and a summary
    m = MODELS.loc[name]
    sub = Path(m.family) / m.params / PROMPT_SET
    out_dir = Path('outputs/tts') / sub
    out_dir.mkdir(parents=True, exist_ok=True)
    sys_dir, model_dir = setup_output_directory(str(Path('metrics_analysis/tts') / sub))
    model, load_s = load_model(name)
    synthesize(model, name, PROMPTS.prompt[0], voice_kwargs(name, PROMPTS.description[0]))   # warm-up
    rows, peak = [], 0.0
    for p in PROMPTS.itertuples():
        kw = voice_kwargs(name, p.description)
        mx.reset_peak_memory()
        metrics_monitor_start(idx=p.id, metrics_folder=sys_dir)
        audio, rate, total, first = synthesize(model, name, p.prompt, kw)
        metrics_monitor_stop()
        peak = max(peak, mx.get_peak_memory() / 1e9)
        wav = out_dir / f'{name}_audio_{p.id}.wav'
        sf.write(wav, audio, rate, subtype='PCM_16')
        rows.append({
            'prompt_id': p.id, 'model': name, 'family': m.family, 'size': m['size'],
            'voice': VOICES[m.family][bool(re.search(r'\bfemale\b|\bwoman\b', p.description.lower()))],
            'description_used': 'full description' if 'instruct' in kw else 'no' if m.family == 'OuteTTS' else 'gender only',
            'input_words': len(p.prompt.split()), 'input_characters': len(p.prompt),
            'output_audio (s)': round(len(audio) / rate, 2), 'sample_rate (Hz)': rate,
            'time_to_first_audio (ms)': round(1000 * first, 1),
            'inference_rate (audio s/s)': round(len(audio) / rate / total, 2),
            'inference_time (s)': round(total, 3),
            'audio_file': str(wav), 'prompt': p.prompt, 'description': p.description,
        })
        print(f'  [{p.id}] {len(audio) / rate:5.1f} s audio in {total:5.2f} s (first audio {1000 * first:.0f} ms)')
    del model
    gc.collect()
    mx.clear_cache()

    df = pd.DataFrame(rows)
    df['transcript'] = transcribe_all(df.audio_file)
    errs = [word_errors(p, t) for p, t in zip(df.prompt, df.transcript)]
    df.insert(df.columns.get_loc('audio_file'), 'word_error_rate (%)', [round(100 * e / n, 2) for n, e in errs])
    df.to_csv(Path(model_dir) / f'{name}_prompt_metrics.csv', index=False)
    hw = pd.concat([pd.read_csv(Path(sys_dir) / f'{i}_system_metrics.csv') for i in df.prompt_id])
    audio_s, time_s = df['output_audio (s)'].sum(), df['inference_time (s)'].sum()
    summary = {
        'model': name, 'family': m.family, 'size': m['size'], 'parameters': m.params, 'precision': m.precision,
        'path': m.path, 'source': m.source, 'prompt_set': PROMPT_SET, 'prompts': len(df),
        'model_load (s)': round(load_s, 1),
        'output_audio (s, total)': round(audio_s, 1),
        'word_error_rate (%)': round(100 * sum(e for _, e in errs) / sum(n for n, _ in errs), 2),
        'time_to_first_audio (ms, avg)': round(df['time_to_first_audio (ms)'].mean(), 1),
        'inference_rate (audio s/s, avg)': round(df['inference_rate (audio s/s)'].mean(), 2),
        'inference_time (s, total)': round(time_s, 2),
        'real_time_factor': round(time_s / audio_s, 3),
        'peak_memory (GB)': round(peak, 2),
        'gpu_utilization (%, avg)': round(hw['GPU Utilization (%)'].mean(), 1),
        'gpu_power (W, avg)': round(hw['gpuPower'].mean(), 1),
        'gpu_temperature (C, max)': round(hw['GPU_Temperature(C)'].max(), 1),
        'cpu_utilization (%, avg)': round(hw['CPU_Util(%)'].mean(), 1),
        'cpu_power (W, avg)': round(hw['cpuPower'].mean(), 1),
        'npu_power (W, avg)': round(hw['npuPower'].mean(), 2),
        'system_energy (J)': round(hw['total_system_power'].sum() * 0.1, 1),   # 100 ms samples
    }
    (out_dir / f'{name}_simple_metrics.json').write_text(json.dumps(summary, indent=2))
    return summary, df

summary, per_prompt = run_model(MODEL)
display(per_prompt.drop(columns=['audio_file', 'prompt', 'description', 'transcript']))
pd.Series(summary, name=MODEL)

## 5. Compare the family

Runs every other model of `FAMILY`, each in its own fresh process (this notebook, run for that model), so every model
starts from the same clean GPU state. Set `TTS_COMPARE=0` (or `COMPARE_FAMILY = False` in section 1) to skip.

In [ ]:
results = {MODEL: summary}
if COMPARE_FAMILY:
    for name, m in MODELS[MODELS.family == FAMILY].iterrows():
        if name == MODEL:
            continue
        print(f'{name}: running in a fresh process...')
        env = {**os.environ, 'TTS_FAMILY': m.family, 'TTS_SIZE': m['size'], 'TTS_PROMPTS': PROMPT_SET, 'TTS_COMPARE': '0'}
        with tempfile.TemporaryDirectory() as tmp:
            run = subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'notebook', '--execute',
                                  'notebooks/tts_text_to_speech.ipynb', '--ExecutePreprocessor.timeout=-1',
                                  '--output-dir', tmp], env=env, capture_output=True, text=True)
        if run.returncode != 0:
            print(f'  failed:\n{run.stderr[-2000:]}')
            continue
        results[name] = json.loads((Path('outputs/tts') / m.family / m.params / PROMPT_SET /
                                    f'{name}_simple_metrics.json').read_text())
        print(f"  WER {results[name]['word_error_rate (%)']}%, {results[name]['inference_rate (audio s/s, avg)']} audio s/s")
pd.DataFrame({name: results[name] for name in MODELS.index if name in results})

## 6. All models run so far

Combines every model run by this notebook on `PROMPT_SET` into `outputs/tts/all_models_<prompt set>.csv` and their per-prompt
metrics into `metrics_analysis/tts/all_models_prompt_metrics_<prompt set>.csv`.

In [ ]:
summaries = [json.loads(f.read_text()) for f in Path('outputs/tts').glob(f'*/*/{PROMPT_SET}/*_simple_metrics.json')]
summaries = sorted((s for s in summaries if 'path' in s),   # results from this notebook (not the earlier one)
                    key=lambda s: list(MODELS.index).index(s['model']))
all_models = pd.DataFrame(summaries)
all_models.to_csv(Path('outputs/tts') / f'all_models_{PROMPT_SET}.csv', index=False)
all_prompts = pd.concat([
    pd.read_csv(Path('metrics_analysis/tts') / s['family'] / s['parameters'] / PROMPT_SET / 'Model_Level_Metrics' /
                f"{s['model']}_prompt_metrics.csv", dtype={'prompt_id': str})
    for s in summaries])
all_prompts.to_csv(Path('metrics_analysis/tts') / f'all_models_prompt_metrics_{PROMPT_SET}.csv', index=False)
print(f'{len(all_models)} models, {len(all_prompts)} prompt rows')
all_models